# 109장 · 소유자 권한과 응답 노출 검증

소유자 권한 검사는 다른 사용자의 정보를 노출하지 않으면서 정당한 소유자의 조회는 허용해야 합니다. 두 경우의 응답 범위를 함께 검증합니다.

가상의 읽기 전용 기록에서 세션 소유자와 자료 접근 범위를 검수합니다. 거부 응답의 노출 필드와 정상 사용자의 조회 결과를 각각 확인합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 팀별 변형을 시도한 뒤 해당 셀 아래에서 예시를 펼쳐 비교할 수 있습니다.


## 공개 응답에 허용한 필드만 남기기

108장의 평가 기록을 가진 팀이 소유자 확인에 실패한 경우를 살펴봅니다. 이미 거부된 예시 기록을 화면으로 옮길 때는 허용한 오류 필드만 구성합니다.

이 프로젝트의 노트북은 한 폴더에서 열어 주세요. 첫 셀은 `team-project/final/chapter108.json`을 먼저 찾습니다. 팀 파일이 없으면 [이 장의 시작 예시](chapter109-start.json)를 노트북 옆에 내려받으세요. 출력 첫 줄에서 어느 파일을 읽었는지 확인할 수 있습니다.

In [ ]:
from pathlib import Path
import json
import re

here = Path.cwd()
project_root = here.parent if re.fullmatch(r"chapter[0-9]+", here.name) else here
team_file = project_root / "team-project/final/chapter108.json"
sample_file = here / "chapter109-start.json"
input_file = team_file if team_file.is_file() else sample_file
if not input_file.is_file():
    raise FileNotFoundError("이 장의 시작 예시를 내려받아 노트북 옆에 두세요.")
incoming = json.loads(input_file.read_text(encoding="utf-8"))
print("팀 기록" if input_file == team_file else "교재 예시", input_file.name)
print(incoming['answer_review'])

교재 예시 chapter109-start.json
{'doc': True, 'fact': True, 'case': 'Q01', 'split': '공개 개발 예시'}


두 번째 줄은 읽은 파일의 `answer_review` 값입니다.

## 다른 사용자의 대화 조회 거부

방어 점검 기록　상황 가상 사용자 bob이 자신에게 없는 대화를 조회 / 기대 본문·소유자·존재 여부를 노출하지 않음 / 로컬 관찰 session_not_found, 저장 시도 없음 / 후속 자기 대화는 그대로 읽을 수 있는지 대조. 이 검증에는 수업용 가명과 가상 문서만 사용했습니다.


### 셀 01 · 조회 거부 응답의 노출 범위

other-owner 사례의 관찰 필드를 읽습니다. 서버 요청을 보내지 않고 이미 기록된 결과만 확인합니다.

정상 사례를 기록할 때는 가상 사용자와 그 사용자에게 허용한 세션을 한 쌍으로 적으세요. 접근을 거부한 화면에는 안내만 남기고, 어느 권한 검사에서 멈췄는지는 내부 점검 기록에 보관합니다.


In [1]:
observed = {'boundary_status': 'rejected',
 'result': None,
 'storage_status': 'not_attempted',
 'error': {'code': 'session_not_found',
           'stage': 'session_read',
           'error_type': 'LookupError'}}
print(observed["boundary_status"], observed["error"]["code"])
print(observed["result"], observed["storage_status"])

rejected session_not_found
None not_attempted


rejected와 session_not_found가 나오고 결과는 None, 저장은 not_attempted입니다. 다른 사람이 가진 대화인지 존재하지 않는 대화인지에 관한 정보도 사용자 안내에 덧붙이지 않습니다.

다른 소유자의 세션을 요청했을 때 응답에서 문서 ID나 과거 질문까지 드러나면 본문을 숨긴 효과가 사라집니다. 거부 응답의 상태뿐 아니라 외부에 반환한 필드를 확인하고, 정상 소유자 경로에서는 자신의 자료를 읽을 수 있는지도 따로 검사합니다.


#### 다른 소유자의 세션을 조회하면

<mark>접근을 거절할 때 답변 본문과 출처도 함께 살펴봅니다.</mark> 거절 상태를 반환하고도 세션 존재나 문서 ID가 응답에 남을 수 있습니다. 같은 데이터로 정상 소유자 요청도 실행해 정상 경로가 유지되는지 확인하세요.

선수 개념 · 95장 검색 모듈의 권한 필터와 빈 결과

## 정상 소유자의 조회 회귀 검사

수정 제안 예시　세션 자료를 읽기 전에 확인된 사용자와 세션 조합을 사용합니다. 정상 회귀는 bob 자신의 대화에 장비 B 질문을 보냈을 때 RULE-B가 여전히 나오는지 보는 것입니다. 외부 쓰기 도구를 채택하지 않은 팀은 외부 변경이 성공적으로 차단됐다고 적기보다 그 기능을 이번 시험에서 다루지 않았다고 기록합니다.


### 팀 참고 예시 · 정상 소유자 응답


In [2]:
normal_result = {'status': 'answered',
 'answer': '점검 전 고정 장치를 확인합니다.',
 'sources': [{'id': 'RULE-B',
              'text': '점검 전 고정 장치를 확인합니다.',
              'source': '가상 장비실 규정 B',
              'page': 2,
              'section': '2.2',
              'revised_on': '2026-11-01',
              'version': 'rules-v1'}],
 'mode': 'local_rule_fixture',
 'model_calls': 0}
print(normal_result["status"], normal_result["sources"][0]["id"])

answered RULE-B


answered와 RULE-B가 나옵니다. 자신의 기록은 읽을 수 있으므로 앞의 거부 사례와 함께 읽어야 경계의 효과를 설명할 수 있습니다.


#### 조회 거부 뒤 정상 사용자의 경로도 검증하기

```mermaid
flowchart TD
nrequest["bob의 요청"]
  nowner["소유자·세션"]
  nrejected["조회 거부"]
  nown["자기 대화"]
  nresult["정상 결과"]
  nrequest --> nowner
  nowner --> nrejected
  nowner --> nown
  nown --> nresult
```

접근을 거부한 검사 다음에 bob 자신의 대화로 정상 사례를 확인합니다. 이 과제에는 정상 응답 자료가 제공됩니다.

## 거부 응답에서 공개할 필드를 좁히기

가상 소유자 세션 조회가 거부되면 운영 기록에는 원인 추적 정보가 남습니다. 이용자에게 보낼 본문에는 공통 오류 안내만 남기도록, 서버 기록에서 새 공개 응답을 구성합니다.

### 셀 04 · 이미 거부된 두 가상 기록을 같은 안내로 바꾸기

이 실습은 저장된 가상 거부 기록을 화면용으로 바꾸는 방어 코드입니다. 실제 사용자 세션에 접속하지 않습니다.

In [3]:
def public_session_error(audit_record):
    if audit_record["boundary_status"] != "rejected":
        raise ValueError("거부된 기록만 전달하세요.")
    return {"code": "session_not_found",
            "message": "현재 대화를 확인할 수 없습니다. 선택한 대화를 확인해 주세요."}

another_record = {**observed, "internal_reason": "no matching session"}
display_a = public_session_error(observed)
display_b = public_session_error(another_record)
print(display_a == display_b, list(display_a))
print(display_a["message"])

True ['code', 'message']
현재 대화를 확인할 수 없습니다. 선택한 대화를 확인해 주세요.


내부 기록의 모양이 달라도 공개한 두 필드는 같습니다. 이 함수는 이미 거부한 기록을 가공합니다. 누가 어느 세션을 읽을 수 있는지 결정하는 서버 검사까지 수행한 것은 아닙니다.

### 정상 사용자의 회귀 확인

거부 사례의 공개 문구를 바꾼 다음에는 bob이 자기 대화에서 정상적으로 받은 normal_result도 다시 확인하세요. 정상 응답까지 일괄해서 오류 문구로 바꾸면 노출은 줄어도 정당한 사용자가 서비스를 이용할 수 없습니다.

<mark>사용자에게 보일 필드는 내부 기록을 복사한 뒤 지우기보다 허용한 값으로 새로 구성할 수 있습니다.</mark> 운영 기록에 나중에 내부 사유 필드가 추가되어도 위 함수의 공개 JSON에는 두 필드만 들어갑니다.

110장의 측정 로그도 먼저 필요한 필드를 정하고, 질문 본문·세션 소유자의 불필요한 사본을 만들지 않습니다.

## 팀 사례에 적용하기 · 공개 응답에 허용한 필드만 남기기

위 예제에 첫 셀에서 읽어 둔 팀 기록을 넣어 봅니다.

In [ ]:
internal = {**observed, "request_id": incoming["reply"]["request_id"],
            "evaluation": incoming["answer_review"]}
public_error = public_session_error(internal)
new = {**incoming, "public_error": public_error,
       "visible_keys": sorted(public_error), "evaluation_exposed": "evaluation" in public_error}
print(new["visible_keys"], new["evaluation_exposed"])

['code', 'message'] False


사용자 메시지에는 내부 요청 ID와 평가 결과가 없습니다. 팀의 API가 소유자를 실제로 확인하는지는 공개 메시지의 모양과 따로 검증해야 합니다.

In [ ]:
output_file = project_root / "team-project/final/chapter109.json"
output_file.parent.mkdir(parents=True, exist_ok=True)
with output_file.open("a+", encoding="utf-8") as handle:
    handle.seek(0)
    previous_text = handle.read()
    if previous_text and json.loads(previous_text) != new:
        raise FileExistsError("같은 장의 다른 결과가 있습니다. 기존 JSON을 다른 이름으로 보관한 뒤 다시 실행하세요.")
    if not previous_text:
        json.dump(new, handle, ensure_ascii=False, indent=2)
        handle.write("\n")
print(output_file.relative_to(project_root).as_posix())

team-project/final/chapter109.json


110장 첫 셀이 이 파일을 읽습니다. 입력을 바꿔 다시 실행할 때는 기존 결과 파일의 이름을 바꿔 보관하세요.

## 돌아보기

다음 단계에는 어떤 경계가 로컬 예제에서 지켜졌고 실제 서비스에서는 무엇을 별도로 확인해야 하는지 넘기세요. 여기에는 실제 인증·외부 모델·운영 보안 사고에 대한 관측이 없습니다.
